# Notebook 02c-UNSW — Retrain DNNs with longer training

**Purpose:** Notebook 02b diagnostic showed `dnn_5class_smote` was prematurely stopped (best val_loss at epoch 58/60 — still improving). The `dnn_binary_cw` model was trained with the same patience=5 setting and likely has the same issue. We retrain both with:

- `max_epochs = 80` (was 40)
- `patience = 10` (was 5)

Everything else (architecture, optimizer, batch size, learning rate, SMOTE, class weights) is unchanged.

## What this notebook does

1. Retrains `dnn_binary_cw` with the new settings
2. Retrains `dnn_5class_smote` with the new settings
3. **Overwrites** the model files, prediction arrays, probability arrays, and the corresponding entries in `metrics.json`
4. Backs up the old metrics.json first, just in case
5. Regenerates the comparison table and confusion-matrix figure
6. Commits and pushes

**The RF and XGBoost models are not touched.** Only the two DNNs are retrained.

## 1. Environment setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, shutil
shutil.copy('/content/drive/MyDrive/XIDS_Research/.gitconfig',       '/root/.gitconfig')
shutil.copy('/content/drive/MyDrive/XIDS_Research/.git-credentials', '/root/.git-credentials')

PROJECT_DIR = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(PROJECT_DIR)

!git config --global credential.helper 'store --file /content/drive/MyDrive/XIDS_Research/.git-credentials'
!git pull origin main

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
!nvidia-smi -L

From https://github.com/anasbiswas1/xids-research
 * branch            main       -> FETCH_HEAD
Already up to date.
Device: cuda
GPU 0: Tesla T4 (UUID: GPU-8558d3ee-f947-9e01-e8c9-7585d9fd4e07)


## 2. Load data, existing metrics, and back up

In [3]:
import numpy as np
import json
from pathlib import Path
from collections import Counter

REPO       = Path(PROJECT_DIR)
PROC_DIR   = REPO / 'data/processed/unsw_nb15'
MODELS_DIR = REPO / 'models/unsw_nb15'
PREDS_DIR  = MODELS_DIR / 'predictions'
PROBS_DIR  = MODELS_DIR / 'probabilities'

# Load data
X_train = np.load(PROC_DIR / 'X_train.npy')
X_test  = np.load(PROC_DIR / 'X_test.npy')
y_train_binary = np.load(PROC_DIR / 'y_train_binary.npy')
y_test_binary  = np.load(PROC_DIR / 'y_test_binary.npy')
y_train_5class = np.load(PROC_DIR / 'y_train_5class.npy')
y_test_5class  = np.load(PROC_DIR / 'y_test_5class.npy')

FIVE_CLASS_NAMES = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']

# Load existing metrics and back up
with open(MODELS_DIR / 'metrics.json') as f:
    ALL_METRICS = json.load(f)

import time
ts = time.strftime('%Y%m%d_%H%M%S')
shutil.copy(MODELS_DIR / 'metrics.json', MODELS_DIR / f'metrics_backup_{ts}.json')
print(f'Backed up old metrics to: metrics_backup_{ts}.json')

print(f'\nExisting models in metrics.json: {list(ALL_METRICS.keys())}')
print(f'\nOld DNN scores (to beat):')
print(f"  dnn_binary_cw:    F1m={ALL_METRICS['dnn_binary_cw']['f1_macro']:.4f}  MCC={ALL_METRICS['dnn_binary_cw']['mcc']:.4f}")
print(f"  dnn_5class_smote: F1m={ALL_METRICS['dnn_5class_smote']['f1_macro']:.4f}  U2R F1={ALL_METRICS['dnn_5class_smote']['per_class_f1']['U2R']:.4f}")

Backed up old metrics to: metrics_backup_20260526_180415.json

Existing models in metrics.json: ['rf_binary_cw', 'xgb_binary_cw', 'dnn_binary_cw', 'rf_5class_smote', 'xgb_5class_smote', 'dnn_5class_smote']

Old DNN scores (to beat):
  dnn_binary_cw:    F1m=0.8320  MCC=0.6981
  dnn_5class_smote: F1m=0.5055  U2R F1=0.2200


## 3. SMOTE for 5-class (regenerate identically to Notebook 02)

In [4]:
from imblearn.over_sampling import SMOTE

pre = Counter(y_train_5class)
largest_attack = max(pre[c] for c in [1, 2, 3, 4])
sampling_strategy = {0: pre[0]}
for c in [1, 2, 3, 4]:
    sampling_strategy[c] = max(pre[c], largest_attack)

smote = SMOTE(sampling_strategy=sampling_strategy, random_state=42, k_neighbors=5)
X_train_sm, y_train_5class_sm = smote.fit_resample(X_train, y_train_5class)
X_train_sm = X_train_sm.astype(np.float32)

print(f'Post-SMOTE shape: {X_train_sm.shape}')

Post-SMOTE shape: (269292, 196)


## 4. DNN architecture and trainer (with longer settings)

In [5]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

class XIDSMLP(nn.Module):
    def __init__(self, n_features, n_classes, p_drop=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(256, 128),        nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(128, 64),         nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(64, 32),          nn.BatchNorm1d(32),  nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(32, n_classes),
        )
    def forward(self, x):
        return self.net(x)


def train_dnn_long(X_tr, y_tr, X_te, n_classes, class_weights=None,
                   epochs=80, batch_size=512, lr=1e-3, patience=10, seed=42):
    """Same trainer as Notebook 02 but with longer epochs / patience defaults."""
    torch.manual_seed(seed)

    X_tr2, X_val, y_tr2, y_val = train_test_split(
        X_tr, y_tr, test_size=0.1, random_state=seed, stratify=y_tr,
    )

    Xt  = torch.tensor(X_tr2, dtype=torch.float32)
    yt  = torch.tensor(y_tr2, dtype=torch.long)
    Xv  = torch.tensor(X_val, dtype=torch.float32).to(device)
    yv  = torch.tensor(y_val, dtype=torch.long).to(device)
    Xte = torch.tensor(X_te,  dtype=torch.float32).to(device)

    loader = DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=True)

    model = XIDSMLP(X_tr.shape[1], n_classes).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)

    if class_weights is not None:
        cw_t = torch.tensor(class_weights, dtype=torch.float32, device=device)
        criterion = nn.CrossEntropyLoss(weight=cw_t)
    else:
        criterion = nn.CrossEntropyLoss()

    best_val_loss = float('inf')
    best_state    = None
    best_epoch    = 0
    no_improve    = 0

    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xv)
            val_loss   = criterion(val_logits, yv).item()
            val_acc    = (val_logits.argmax(1) == yv).float().mean().item()

        if ep == 1 or ep % 5 == 0:
            print(f'  ep {ep:>3d}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}')

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_epoch    = ep
            best_state    = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve    = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stop at epoch {ep} (best={best_epoch})')
                break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        logits  = model(Xte)
        y_proba = F.softmax(logits, dim=1).cpu().numpy()
        y_pred  = y_proba.argmax(axis=1)

    return model, y_pred, y_proba, best_val_loss, best_epoch

print('Trainer ready.')

Trainer ready.


## 5. Shared evaluator (matches Notebook 02 exactly)

In [6]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, confusion_matrix, roc_auc_score,
)

def evaluate(model_name, y_true, y_pred, y_proba, target_type):
    metrics = {
        'model':        model_name,
        'target_type':  target_type,
        'accuracy':     float(accuracy_score(y_true, y_pred)),
        'precision_macro': float(precision_score(y_true, y_pred, average='macro', zero_division=0)),
        'recall_macro':    float(recall_score(y_true, y_pred, average='macro', zero_division=0)),
        'f1_macro':        float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'f1_weighted':     float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'mcc':             float(matthews_corrcoef(y_true, y_pred)),
    }
    if target_type == 'binary':
        pos_proba = y_proba[:, 1] if y_proba.ndim == 2 else y_proba
        metrics['auc_roc'] = float(roc_auc_score(y_true, pos_proba))
        metrics['per_class_f1'] = {
            'Normal': float(f1_score(y_true, y_pred, pos_label=0)),
            'Attack': float(f1_score(y_true, y_pred, pos_label=1)),
        }
    else:
        per_class = f1_score(y_true, y_pred, average=None, labels=range(5), zero_division=0)
        metrics['per_class_f1'] = {name: float(per_class[i]) for i, name in enumerate(FIVE_CLASS_NAMES)}
    metrics['confusion_matrix'] = confusion_matrix(y_true, y_pred).tolist()
    return metrics

## 6. Retrain dnn_binary_cw

In [7]:
name = 'dnn_binary_cw'
print(f'Retraining {name} with epochs=80, patience=10...')
t0 = time.time()

cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train_binary)
print(f'  class weights: Normal={cw[0]:.3f}, Attack={cw[1]:.3f}')

model, y_pred, y_proba, best_val, best_ep = train_dnn_long(
    X_train, y_train_binary, X_test,
    n_classes=2, class_weights=cw,
    epochs=80, patience=10,
)

metrics = evaluate(name, y_test_binary, y_pred, y_proba, 'binary')
metrics['train_time_sec'] = round(time.time() - t0, 1)
metrics['retrained'] = True
metrics['best_epoch'] = best_ep

# Compare to old
old = ALL_METRICS[name]
print(f'\n{name} retrain results:')
print(f"             OLD       NEW        Δ")
for k in ['accuracy', 'f1_macro', 'mcc', 'auc_roc']:
    delta = metrics[k] - old[k]
    flag = ' ↑' if delta > 0.001 else (' ↓' if delta < -0.001 else '')
    print(f"  {k:12s} {old[k]:.4f}    {metrics[k]:.4f}   {delta:+.4f}{flag}")
print(f"  best_epoch: {best_ep}, time: {metrics['train_time_sec']}s")

ALL_METRICS[name] = metrics

# Persist
torch.save(model.state_dict(), MODELS_DIR / f'{name}.pt')
np.save(PREDS_DIR / f'{name}_preds.npy', y_pred)
np.save(PROBS_DIR / f'{name}_proba.npy', y_proba)
with open(MODELS_DIR / 'metrics.json', 'w') as f:
    json.dump(ALL_METRICS, f, indent=2, default=str)
print('Saved.')

Retraining dnn_binary_cw with epochs=80, patience=10...
  class weights: Normal=1.208, Attack=0.853
  ep   1  val_loss=0.1666  val_acc=0.9236
  ep   5  val_loss=0.1597  val_acc=0.9269
  ep  10  val_loss=0.1637  val_acc=0.9180
  ep  15  val_loss=0.1566  val_acc=0.9253
  ep  20  val_loss=0.1482  val_acc=0.9274
  ep  25  val_loss=0.1506  val_acc=0.9277
  ep  30  val_loss=0.1505  val_acc=0.9297
  Early stop at epoch 30 (best=20)

dnn_binary_cw retrain results:
             OLD       NEW        Δ
  accuracy     0.8321    0.8417   +0.0096 ↑
  f1_macro     0.8320    0.8412   +0.0093 ↑
  mcc          0.6981    0.7041   +0.0060 ↑
  auc_roc      0.9655    0.9637   -0.0018 ↓
  best_epoch: 20, time: 56.5s
Saved.


## 7. Retrain dnn_5class_smote

In [ ]:
name = 'dnn_5class_smote'
print(f'Retraining {name} with epochs=80, patience=10...')
t0 = time.time()

cw5 = compute_class_weight('balanced', classes=np.arange(5), y=y_train_5class_sm)
print(f'  class weights: {dict(zip(FIVE_CLASS_NAMES, cw5.round(3)))}')

model, y_pred, y_proba, best_val, best_ep = train_dnn_long(
    X_train_sm, y_train_5class_sm, X_test,
    n_classes=5, class_weights=cw5,
    epochs=80, patience=10,
)

metrics = evaluate(name, y_test_5class, y_pred, y_proba, 'multiclass')
metrics['train_time_sec'] = round(time.time() - t0, 1)
metrics['retrained'] = True
metrics['best_epoch'] = best_ep

# Compare to old
old = ALL_METRICS[name]
print(f'\n{name} retrain results:')
print(f"               OLD       NEW        Δ")
for k in ['accuracy', 'f1_macro', 'mcc']:
    delta = metrics[k] - old[k]
    flag = ' ↑' if delta > 0.001 else (' ↓' if delta < -0.001 else '')
    print(f"  {k:14s} {old[k]:.4f}    {metrics[k]:.4f}   {delta:+.4f}{flag}")
for cn in FIVE_CLASS_NAMES:
    delta = metrics['per_class_f1'][cn] - old['per_class_f1'][cn]
    flag = ' ↑' if delta > 0.001 else (' ↓' if delta < -0.001 else '')
    print(f"  F1[{cn:6s}]:   {old['per_class_f1'][cn]:.4f}    {metrics['per_class_f1'][cn]:.4f}   {delta:+.4f}{flag}")
print(f"  best_epoch: {best_ep}, time: {metrics['train_time_sec']}s")

ALL_METRICS[name] = metrics

# Persist
torch.save(model.state_dict(), MODELS_DIR / f'{name}.pt')
np.save(PREDS_DIR / f'{name}_preds.npy', y_pred)
np.save(PROBS_DIR / f'{name}_proba.npy', y_proba)
with open(MODELS_DIR / 'metrics.json', 'w') as f:
    json.dump(ALL_METRICS, f, indent=2, default=str)
print('Saved.')

Retraining dnn_5class_smote with epochs=80, patience=10...
  class weights: {'Normal': np.float64(0.962), 'DoS': np.float64(1.01), 'Probe': np.float64(1.01), 'R2L': np.float64(1.01), 'U2R': np.float64(1.01)}
  ep   1  val_loss=0.7031  val_acc=0.7216
  ep   5  val_loss=0.5642  val_acc=0.7702
  ep  10  val_loss=0.5355  val_acc=0.7835
  ep  15  val_loss=0.5313  val_acc=0.7804
  ep  20  val_loss=0.5156  val_acc=0.7880
  ep  25  val_loss=0.5214  val_acc=0.7838
  ep  30  val_loss=0.5140  val_acc=0.7881
  ep  35  val_loss=0.5023  val_acc=0.7895
  ep  40  val_loss=0.5011  val_acc=0.7898
  ep  45  val_loss=0.5019  val_acc=0.7918
  ep  50  val_loss=0.4928  val_acc=0.7946
  ep  55  val_loss=0.4935  val_acc=0.7968
  ep  60  val_loss=0.4895  val_acc=0.7962
  ep  65  val_loss=0.4876  val_acc=0.7962
  ep  70  val_loss=0.4852  val_acc=0.7968


## 8. Regenerate comparison table and confusion matrices

In [ ]:
import pandas as pd

rows = []
for n, m in ALL_METRICS.items():
    row = {
        'Model':       n,
        'Target':      m['target_type'],
        'Accuracy':    round(m['accuracy'], 4),
        'F1-macro':    round(m['f1_macro'], 4),
        'F1-weighted': round(m['f1_weighted'], 4),
        'MCC':         round(m['mcc'], 4),
    }
    if m['target_type'] == 'binary':
        row['AUC-ROC'] = round(m['auc_roc'], 4)
        row['U2R F1']  = None
    else:
        row['AUC-ROC'] = None
        row['U2R F1']  = round(m['per_class_f1']['U2R'], 4)
    row['Time (s)'] = m.get('train_time_sec', None)
    rows.append(row)

comparison = pd.DataFrame(rows)
comparison_path = REPO / 'results/tables/unsw_model_comparison.csv'
comparison.to_csv(comparison_path, index=False)

print(comparison.to_string(index=False))
print(f'\nSaved: {comparison_path}')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
model_order = [
    'rf_binary_cw', 'xgb_binary_cw', 'dnn_binary_cw',
    'rf_5class_smote', 'xgb_5class_smote', 'dnn_5class_smote',
]

for ax, n in zip(axes.flat, model_order):
    cm = np.array(ALL_METRICS[n]['confusion_matrix'])
    labels = ['Normal', 'Attack'] if ALL_METRICS[n]['target_type'] == 'binary' else FIVE_CLASS_NAMES

    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f"{n}\nAcc={ALL_METRICS[n]['accuracy']:.3f}, F1m={ALL_METRICS[n]['f1_macro']:.3f}", fontsize=10)

    for i in range(len(labels)):
        for j in range(len(labels)):
            colour = 'white' if cm_norm[i, j] > 0.5 else 'black'
            ax.text(j, i, f'{cm_norm[i, j]:.2f}', ha='center', va='center', fontsize=8, color=colour)

plt.suptitle('UNSW-NB15 — Confusion matrices (after DNN retrain)', fontsize=14, y=1.00)
plt.tight_layout()

cm_path = REPO / 'results/figures/unsw_confusion_matrices.png'
plt.savefig(cm_path, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {cm_path}')

## 9. Commit and push

In [ ]:
os.chdir(PROJECT_DIR)
!git add notebooks/02c_unsw_dnn_retrain.ipynb \
         results/figures/unsw_confusion_matrices.png \
         results/tables/unsw_model_comparison.csv
!git commit -m "Notebook 02c-UNSW: retrain both DNNs with epochs=80, patience=10 to ensure convergence"
!git push origin main

---

## What to check

1. Both retrains should print `best_epoch` *not* at the very last epoch — that confirms convergence rather than another premature stop.
2. F1-macro should be at-or-above the old value on both DNNs. Even if the gain is small, the model is methodologically cleaner.
3. If a retrain has a *lower* F1 than the old version, that's stochastic noise — but tell me before we move on.

## Next step

**Notebook 03-UNSW** — per-class isotonic calibration. All six models are now converged and ready to calibrate.